In [0]:
from pyspark.sql.functions import *

In [0]:
documents = [
    {
        "doc_id": 1,
        "text": "Employees are entitled to 20 days of paid leave per year. Sick leave is 10 days."
    },
    {
        "doc_id": 2,
        "text": "Procurement requests must be approved by the department head before purchase."
    },
    {
        "doc_id": 3,
        "text": "Dispatch SLA is 48 hours from order confirmation."
    }
]

documents


In [0]:
df = spark.createDataFrame(documents)

df.display()

In [0]:
def simple_chunk(text, chunk_size=80):
    chunks = []
    for i in range(0, len(text), chunk_size):
        chunks.append(text[i:i+chunk_size])
    return chunks

from pyspark.sql.functions import udf
from pyspark.sql.types import ArrayType, StringType

chunk_udf = udf(simple_chunk, ArrayType(StringType()))

#udf arguments should be column. And the data type should be whatever we return from the function
df_chunked = df.withColumn("chunks", chunk_udf(df["text"]))

df_chunked.display()

In [0]:
df_final = df_chunked.select(
    "doc_id",
    explode("chunks").alias("chunk_text")
)

df_final.display()

In [0]:
import mlflow.deployments

client = mlflow.deployments.get_deploy_client("databricks")

response = client.predict(
    endpoint="databricks-bge-large-en",
    inputs={
        "input": ["Employees are entitled to 20 days of paid leave."]
    }
)

for item in response['data']:
    print(item['embedding'])


In [0]:
import mlflow.deployments

client = mlflow.deployments.get_deploy_client("databricks")

def get_embeddings_batch(text_list):
    response = client.predict(
        endpoint="databricks-bge-large-en",
        inputs={"input": text_list}
    )
    return [item["embedding"] for item in response["data"]]

In [0]:
pdf = df_final.toPandas()

In [0]:
pdf

In [0]:
pdf["embedding"] = get_embeddings_batch(pdf["chunk_text"].tolist())

In [0]:
pdf

In [0]:
df_embeddings = spark.createDataFrame(pdf)
df_embeddings.display()

In [0]:
df_embeddings.write.mode("overwrite").saveAsTable("practice.api.rag_documents")



In [0]:
%sql
select * from practice.api.rag_documents limit 10

In [0]:
question = "How many leave days do employees get?"

question_embedding = get_embeddings_batch([question])[0]

len(question_embedding)


In [0]:
import numpy as np

def cosine_similarity(vec1, vec2):
    vec1 = np.array(vec1)
    vec2 = np.array(vec2)
    return np.dot(vec1, vec2) / (np.linalg.norm(vec1) * np.linalg.norm(vec2))

In [0]:
pdf["similarity"] = pdf["embedding"].apply(
    lambda x: cosine_similarity(x, question_embedding)
)

pdf.sort_values("similarity", ascending=False).head()

In [0]:
top_chunk = pdf.sort_values("similarity", ascending=False).iloc[0]["chunk_text"]

top_chunk

In [0]:
databricks-meta-llama-3-70b-instruct

In [0]:
import mlflow.deployments

client = mlflow.deployments.get_deploy_client("databricks")

prompt = f"""
You are an assistant answering questions based ONLY on the provided context.

Context:
{top_chunk}

Question:
{question}

Answer clearly and concisely.
"""

response = client.predict(
    endpoint="databricks-meta-llama-3-3-70b-instruct",
    inputs={
        "messages": [
            {"role": "user", "content": prompt}
        ]
    }
)

response


In [0]:
def ask_rag(question):

    # Step 1: Embed question
    question_embedding = get_embeddings_batch([question])[0]

    # Step 2: Compute similarity
    pdf["similarity"] = pdf["embedding"].apply(
        lambda x: cosine_similarity(x, question_embedding)
    )

    # Step 3: Retrieve top chunk
    top_chunk = pdf.sort_values("similarity", ascending=False).iloc[0]["chunk_text"]

    # Step 4: Create prompt
    prompt = f"""
    You are an assistant answering questions based ONLY on the provided context. Just provide answer straight. Do not provide extra info

    Context:
    {top_chunk}

    Question:
    {question}

    """

    # Step 5: Call LLM
    response = client.predict(
        endpoint="databricks-meta-llama-3-3-70b-instruct",
        inputs={
            "messages": [
                {"role": "user", "content": prompt}
            ]
        }
    )

    return response["choices"][0]["message"]["content"]


In [0]:
ask_rag("How many sick leaves are there?")